# Probability

Split from `01_probability_and_statistics_python.ipynb` to keep this topic self-contained.

**Navigation:** [Topic overview](./01_probability_and_statistics_python.ipynb) · [Next: Statistics Python](./02_statistics_python.ipynb)


<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook teaches core computational literacy used before any serious bioinformatics analysis: reproducible shell workflows, stats reasoning, and environment hygiene.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Difference between command syntax and conceptual model (what changes state vs what only inspects).
- Interpreting statistical output: p-value alone is not enough without effect size and assumptions.
- Reproducibility: the same command can yield different outputs if input files or working directory differ.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


# Probability and Statistics with Python

**Tier 0 -- Computational Foundations | Module 7**

---

## Overview

Module 6 introduced the *concepts* of probability and statistics. This module puts them into practice with Python. You will use **scipy.stats**, **numpy**, **pandas**, and **statsmodels** to implement every test and technique on realistic synthetic biological data.

### Learning Objectives

By the end of this module you will be able to:
- Use the scipy.stats unified API (`pdf`, `cdf`, `ppf`, `rvs`) for any distribution
- Compute descriptive statistics and generate publication-quality diagnostic plots
- Choose and apply the correct hypothesis test (parametric or non-parametric)
- Run one-way ANOVA and post-hoc Tukey HSD for multi-group comparisons
- Quantify correlation and fit simple and multiple linear regression models
- Build bootstrap confidence intervals and permutation tests from scratch
- Calculate required sample sizes and achieved power for included experiments

---

[← Previous: Biostatistics](../../Tier_0_Computational_Foundations/06_Biostatistics/01_biostatistics.ipynb) | [Next: Python Introduction →](../../Tier_1_Python_for_Bioinformatics/01_Python_Introduction/01_python_introduction.ipynb)

---

In [ ]:
# Core imports used throughout the notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.power import TTestIndPower

# Reproducibility
np.random.seed(42)

# Plot styling
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

---

## 1. Probability Distributions in Python

### The scipy.stats Unified API

`scipy.stats` provides every common distribution through a consistent interface. Once you learn it for one distribution, you know it for all of them:

| Method | What it returns | When to use it |
|--------|----------------|----------------|
| `.pdf(x)` / `.pmf(x)` | Probability density (or mass) at x | Plotting the distribution shape |
| `.cdf(x)` | P(X ≤ x) | Computing tail probabilities |
| `.sf(x)` | P(X > x) = 1 - CDF | One-tailed p-values directly |
| `.ppf(q)` | Quantile: x such that P(X ≤ x) = q | Finding critical values |
| `.rvs(size)` | Random samples from the distribution | Simulation and bootstrap |
| `.fit(data)` | MLE parameter estimates | Fitting to real data |

### Normal Distribution: Gene Expression

Log-transformed gene expression values are approximately normally distributed. Here we model a housekeeping gene measured across 200 samples.

In [ ]:
# Normal distribution: log2 expression of a housekeeping gene
# Parameters chosen to match realistic RNA-seq log2(TPM+1) values
mu_expr = 8.5      # mean log2 expression
sigma_expr = 1.2   # standard deviation across samples

gene_dist = stats.norm(loc=mu_expr, scale=sigma_expr)

# Generate 200 synthetic expression measurements
expression_values = gene_dist.rvs(size=200)

# Key probabilities
prob_above_10 = gene_dist.sf(10)          # P(expression > 10)
percentile_95  = gene_dist.ppf(0.95)      # 95th percentile
cdf_at_7       = gene_dist.cdf(7)         # P(expression <= 7)

print(f"Mean expression:         {mu_expr}")
print(f"P(expression > 10):      {prob_above_10:.4f}")
print(f"95th percentile:         {percentile_95:.3f}")
print(f"P(expression <= 7):      {cdf_at_7:.4f}")

In [ ]:
# Plot PDF and CDF side by side
x = np.linspace(mu_expr - 4*sigma_expr, mu_expr + 4*sigma_expr, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# PDF
ax1.plot(x, gene_dist.pdf(x), 'steelblue', lw=2, label='PDF')
ax1.fill_between(x[x > 10], gene_dist.pdf(x[x > 10]), alpha=0.3, color='tomato', label=f'P(X>10) = {prob_above_10:.3f}')
ax1.axvline(mu_expr, ls='--', color='gray', lw=1)
ax1.set_xlabel('log2 expression')
ax1.set_ylabel('Density')
ax1.set_title('Normal PDF — housekeeping gene')
ax1.legend()

# CDF
ax2.plot(x, gene_dist.cdf(x), 'steelblue', lw=2, label='CDF')
ax2.axhline(0.95, ls='--', color='tomato', lw=1, label=f'95th pct = {percentile_95:.2f}')
ax2.axvline(percentile_95, ls='--', color='tomato', lw=1)
ax2.set_xlabel('log2 expression')
ax2.set_ylabel('P(X ≤ x)')
ax2.set_title('Normal CDF — housekeeping gene')
ax2.legend()

plt.tight_layout()
plt.show()

### Binomial Distribution: Variant Allele Counts

When sequencing a diploid organism at a heterozygous SNP position, each read independently has a 50% chance of carrying the alternate allele. With 30x coverage, the number of alt-supporting reads follows Binomial(n=30, p=0.5). A somatic mutation in a tumor with 20% variant allele frequency follows Binomial(n=30, p=0.2).

In [ ]:
# Binomial: variant allele read counts at 30x coverage
coverage = 30
germline_vaf = 0.5   # expected heterozygous variant allele frequency
somatic_vaf  = 0.2   # somatic mutation in tumor (20% of cells)

germ_dist   = stats.binom(n=coverage, p=germline_vaf)
somatic_dist = stats.binom(n=coverage, p=somatic_vaf)

k_values = np.arange(0, coverage + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(k_values - 0.2, germ_dist.pmf(k_values), width=0.4,
       label=f'Germline (p={germline_vaf})', color='steelblue', alpha=0.8)
ax.bar(k_values + 0.2, somatic_dist.pmf(k_values), width=0.4,
       label=f'Somatic (p={somatic_vaf})', color='tomato', alpha=0.8)
ax.set_xlabel('Alt-supporting reads out of 30')
ax.set_ylabel('Probability')
ax.set_title('Binomial PMF: germline vs somatic variant allele counts')
ax.legend()
plt.tight_layout()
plt.show()

# Probability of observing exactly 6 alt reads for each model
print(f"P(alt=6 | germline model): {germ_dist.pmf(6):.5f}")
print(f"P(alt=6 | somatic model):  {somatic_dist.pmf(6):.5f}")
print(f"Likelihood ratio:          {somatic_dist.pmf(6)/germ_dist.pmf(6):.1f}x more likely under somatic model")

### Poisson Distribution: Mutations per Gene

The number of somatic mutations observed in a gene across cancer samples approximates a Poisson distribution when mutations are rare and independent. The expected count (lambda) depends on gene length and the background mutation rate.

In [ ]:
# Poisson: somatic mutation count per gene across 100 tumor samples
# Background rate: ~1 mutation per 1 Mb, gene = 3 kb => lambda = 0.3
# A driver gene will have higher observed counts
lambda_background = 0.3
lambda_driver     = 3.5   # 10x enrichment — likely driver gene

bg_dist     = stats.poisson(mu=lambda_background)
driver_dist = stats.poisson(mu=lambda_driver)

k = np.arange(0, 12)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k, bg_dist.pmf(k), 'o-', color='steelblue', label=f'Background λ={lambda_background}')
ax.plot(k, driver_dist.pmf(k), 's-', color='tomato', label=f'Driver gene λ={lambda_driver}')
ax.set_xlabel('Mutation count')
ax.set_ylabel('Probability')
ax.set_title('Poisson PMF: somatic mutation counts per gene')
ax.legend()
plt.tight_layout()
plt.show()

# P-value: observing 5+ mutations under background model
pval_5_plus = bg_dist.sf(4)   # P(X > 4) = P(X >= 5)
print(f"P(mutations >= 5 | background): {pval_5_plus:.6f}")
print(f"This gene would be flagged as significantly mutated (p = {pval_5_plus:.2e})")

### Negative Binomial: RNA-seq Overdispersion

Raw RNA-seq read counts are *overdispersed* — their variance exceeds the mean. The Poisson model (variance = mean) underestimates this spread. The **Negative Binomial** adds a dispersion parameter that captures biological variability between replicates. DESeq2 and edgeR both use NB models.

scipy.stats uses the `nbinom(n, p)` parameterisation. To convert from the bioinformatics convention (mean μ, dispersion α where var = μ + αμ²):
- `p = 1 / (1 + α*μ)`
- `n = 1 / α`

In [ ]:
# Negative Binomial: RNA-seq count distribution
# Typical low-expressed gene: mean=50, dispersion=0.1 (variance = 50 + 0.1*50^2 = 300)
mu_rnaseq   = 50
alpha_disp  = 0.1   # dispersion parameter

n_nb = 1 / alpha_disp
p_nb = 1 / (1 + alpha_disp * mu_rnaseq)

nb_dist     = stats.nbinom(n=n_nb, p=p_nb)
pois_dist   = stats.poisson(mu=mu_rnaseq)   # for comparison

print(f"NB mean:     {nb_dist.mean():.1f}")
print(f"NB variance: {nb_dist.var():.1f}")
print(f"Poisson var (=mean): {pois_dist.var():.1f}")
print(f"Overdispersion factor: {nb_dist.var() / nb_dist.mean():.1f}x")

k_rna = np.arange(0, 130)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_rna, pois_dist.pmf(k_rna), color='steelblue', lw=2, label=f'Poisson (μ={mu_rnaseq})')
ax.plot(k_rna, nb_dist.pmf(k_rna), color='tomato', lw=2, label=f'NegBinom (μ={mu_rnaseq}, α={alpha_disp})')
ax.set_xlabel('Read count')
ax.set_ylabel('Probability')
ax.set_title('Poisson vs Negative Binomial: RNA-seq count overdispersion')
ax.legend()
plt.tight_layout()
plt.show()

### Visualising Multiple Distributions Together

A side-by-side comparison of the four distributions we use most often in bioinformatics, each with a biological interpretation.

In [ ]:
# Four-panel distribution gallery
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Normal -- gene expression
x_norm = np.linspace(4, 13, 300)
axes[0, 0].plot(x_norm, stats.norm(8.5, 1.2).pdf(x_norm), 'steelblue', lw=2)
axes[0, 0].set_title('Normal: log2 gene expression')
axes[0, 0].set_xlabel('log2(TPM+1)')

# 2. Binomial -- variant allele reads
k_bin = np.arange(0, 31)
axes[0, 1].bar(k_bin, stats.binom(30, 0.5).pmf(k_bin), color='steelblue', alpha=0.8)
axes[0, 1].set_title('Binomial: alt reads (n=30, p=0.5)')
axes[0, 1].set_xlabel('Alt-supporting reads')

# 3. Poisson -- somatic mutations
k_poi = np.arange(0, 12)
axes[1, 0].bar(k_poi, stats.poisson(2.0).pmf(k_poi), color='steelblue', alpha=0.8)
axes[1, 0].set_title('Poisson: mutation count (λ=2)')
axes[1, 0].set_xlabel('Mutations per gene')

# 4. Negative Binomial -- RNA-seq counts
k_nb = np.arange(0, 130)
axes[1, 1].plot(k_nb, stats.nbinom(10, 10/(10+50)).pmf(k_nb), color='steelblue', lw=2)
axes[1, 1].set_title('NegBinom: RNA-seq counts (μ=50, α=0.1)')
axes[1, 1].set_xlabel('Read count')

for ax in axes.flat:
    ax.set_ylabel('Probability')

plt.suptitle('Probability distributions in bioinformatics', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---

## 2. Descriptive Statistics and Visualization

Before running any test, explore your data visually. The goal is to understand the distribution shape, spot outliers, and decide whether parametric or non-parametric methods are appropriate.

We simulate a gene expression dataset: 60 samples across three tissue types (liver, kidney, brain), with 500 genes each.

In [ ]:
# Generate synthetic expression dataset: 60 samples x 500 genes
n_samples_per_tissue = 20
n_genes = 500
tissues = ['liver', 'kidney', 'brain']

# Each tissue has a slightly different mean expression profile
tissue_offsets = {'liver': 0.0, 'kidney': 0.3, 'brain': -0.2}

expr_data = {}
for tissue in tissues:
    offset = tissue_offsets[tissue]
    # Base expression drawn from N(8, 1.5), tissue-specific shift added
    expr_data[tissue] = np.random.normal(loc=8.0 + offset, scale=1.5,
                                          size=(n_samples_per_tissue, n_genes))

# Focus on one gene (index 0) for univariate analysis
gene_liver  = expr_data['liver'][:, 0]
gene_kidney = expr_data['kidney'][:, 0]
gene_brain  = expr_data['brain'][:, 0]

# Summary statistics with pandas
summary_df = pd.DataFrame({'liver': gene_liver, 'kidney': gene_kidney, 'brain': gene_brain})
print("Summary statistics for gene_0 across tissues:")
print(summary_df.describe().round(3))

In [ ]:
# Histograms + kernel density plots (KDE)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
colors = ['steelblue', 'seagreen', 'tomato']

for ax, (tissue, col) in zip(axes, zip(tissues, colors)):
    data = summary_df[tissue]
    ax.hist(data, bins=10, density=True, alpha=0.5, color=col)
    # Overlay KDE
    kde = stats.gaussian_kde(data)
    x_kde = np.linspace(data.min() - 1, data.max() + 1, 200)
    ax.plot(x_kde, kde(x_kde), color=col, lw=2)
    ax.axvline(data.mean(), ls='--', color='black', lw=1, label=f'mean={data.mean():.2f}')
    ax.set_title(tissue.capitalize())
    ax.set_xlabel('log2 expression')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Density')
plt.suptitle('Histogram + KDE: gene_0 expression per tissue', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for comparing groups — immediately highlights medians, IQR, outliers
fig, ax = plt.subplots(figsize=(8, 5))

bp = ax.boxplot([gene_liver, gene_kidney, gene_brain],
                labels=[t.capitalize() for t in tissues],
                patch_artist=True,
                medianprops=dict(color='black', lw=2))

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

# Overlay individual data points
for i, (data, color) in enumerate(zip([gene_liver, gene_kidney, gene_brain], colors), start=1):
    jitter = np.random.uniform(-0.1, 0.1, size=len(data))
    ax.scatter(np.full(len(data), i) + jitter, data, color=color, alpha=0.4, s=18, zorder=3)

ax.set_ylabel('log2 expression')
ax.set_title('Gene_0 expression: box plot across tissues')
plt.tight_layout()
plt.show()

In [ ]:
# Q-Q plots for normality assessment
# If data are normal, points fall on the diagonal reference line
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, tissue, color in zip(axes, tissues, colors):
    data = summary_df[tissue]
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist='norm')
    ax.scatter(osm, osr, color=color, alpha=0.7, s=25)
    ax.plot(osm, slope * np.array(osm) + intercept, 'k--', lw=1.5, label=f'R²={r**2:.3f}')
    ax.set_title(f'Q-Q: {tissue.capitalize()}')
    ax.set_xlabel('Theoretical quantiles')
    ax.set_ylabel('Sample quantiles')
    ax.legend(fontsize=9)

plt.suptitle('Q-Q plots — assessing normality of gene_0 expression', y=1.02)
plt.tight_layout()
plt.show()
print("Points closely following the diagonal indicate the data are approximately normal.")

---

## 3. Hypothesis Testing

The general workflow for every hypothesis test:

1. **State H₀ and H₁** — null and alternative hypotheses
2. **Check assumptions** — normality, equal variances, independence
3. **Choose the test** — parametric if assumptions hold, non-parametric otherwise
4. **Compute the test statistic and p-value**
5. **Interpret** — reject H₀ if p < α (typically 0.05)

### One-Sample t-Test: Is Expression Different from Control?

**H₀**: The mean expression of a gene in treated samples equals the control mean (μ₀ = 8.0)  
**H₁**: The mean expression is different from 8.0

In [ ]:
# One-sample t-test: gene expression vs control reference value
control_mean = 8.0   # reference expression level in control condition

# Treated samples: true mean is 8.6 (a real effect)
treated_samples = np.random.normal(loc=8.6, scale=1.4, size=25)

t_stat, p_value = stats.ttest_1samp(treated_samples, popmean=control_mean)

print("--- One-Sample t-Test ---")
print(f"Sample mean:      {treated_samples.mean():.3f}")
print(f"Control mean (μ₀): {control_mean}")
print(f"t-statistic:      {t_stat:.4f}")
print(f"p-value:          {p_value:.4f}")
alpha = 0.05
conclusion = "Reject H₀" if p_value < alpha else "Fail to reject H₀"
print(f"Decision (α={alpha}): {conclusion}")

### Two-Sample t-Test and Welch's t-Test: Tumor vs Normal

When comparing two independent groups, use:
- **Student's t-test** (`equal_var=True`): assumes equal population variances
- **Welch's t-test** (`equal_var=False`): does not assume equal variances — preferred by default

In [ ]:
# Two-sample t-test: gene expression in tumor vs normal tissue
n_tumor  = 30
n_normal = 28

# Tumor samples: higher mean expression (oncogene), larger variance
tumor_expr  = np.random.normal(loc=9.8, scale=1.8, size=n_tumor)
normal_expr = np.random.normal(loc=7.9, scale=1.1, size=n_normal)

# Student's t-test (assumes equal variances)
t_student, p_student = stats.ttest_ind(tumor_expr, normal_expr, equal_var=True)

# Welch's t-test (does not assume equal variances — safer default)
t_welch, p_welch = stats.ttest_ind(tumor_expr, normal_expr, equal_var=False)

print(f"Tumor mean:   {tumor_expr.mean():.3f}  | Normal mean:  {normal_expr.mean():.3f}")
print(f"Tumor SD:     {tumor_expr.std():.3f}   | Normal SD:    {normal_expr.std():.3f}")
print()
print(f"Student's t:  t={t_student:.3f}, p={p_student:.4f}")
print(f"Welch's t:    t={t_welch:.3f}, p={p_welch:.4f}")
print()
print("When variances are clearly unequal, Welch's t-test is more accurate.")

### Paired t-Test: Before/After Treatment

When the same sample is measured twice (e.g., before and after drug treatment), the measurements are paired. The paired t-test removes between-sample variability and is more powerful than a two-sample test.

In [ ]:
# Paired t-test: gene expression before and after drug treatment in same patients
n_patients = 20

# Each patient has a baseline expression level
baseline = np.random.normal(loc=8.0, scale=1.5, size=n_patients)
# After treatment: true mean decrease of 0.8 log2 units
treatment_effect = np.random.normal(loc=-0.8, scale=0.5, size=n_patients)
after_treatment = baseline + treatment_effect

t_paired, p_paired = stats.ttest_rel(before=baseline, after=after_treatment)

# Compare with unpaired test on the same data to show power difference
t_unpaired, p_unpaired = stats.ttest_ind(baseline, after_treatment)

print(f"Mean before:    {baseline.mean():.3f}")
print(f"Mean after:     {after_treatment.mean():.3f}")
print(f"Mean difference: {(after_treatment - baseline).mean():.3f}")
print()
print(f"Paired t-test:   t={t_paired:.3f}, p={p_paired:.6f}")
print(f"Unpaired t-test: t={t_unpaired:.3f}, p={p_unpaired:.4f}")
print()
print("The paired test has much higher power (lower p-value) by controlling for patient-to-patient variability.")

### Mann-Whitney U: Non-Parametric Alternative

When your data are not normally distributed, or when you have small samples and cannot verify normality, use the **Mann-Whitney U test** (also called Wilcoxon rank-sum). It compares the rank distributions of two groups rather than their means.

In [ ]:
# Mann-Whitney U: compare protein abundance (skewed distribution) in two cell lines
# Log-normal data — common for protein abundances
cell_line_A = np.random.lognormal(mean=2.5, sigma=0.8, size=18)
cell_line_B = np.random.lognormal(mean=3.0, sigma=0.9, size=20)

# Mann-Whitney U test
u_stat, p_mw = stats.mannwhitneyu(cell_line_A, cell_line_B, alternative='two-sided')

# For comparison: t-test on the raw (non-normal) data
t_stat_raw, p_t_raw = stats.ttest_ind(cell_line_A, cell_line_B)

print(f"Cell line A: median={np.median(cell_line_A):.2f}, mean={cell_line_A.mean():.2f}")
print(f"Cell line B: median={np.median(cell_line_B):.2f}, mean={cell_line_B.mean():.2f}")
print()
print(f"Mann-Whitney U: U={u_stat:.0f}, p={p_mw:.4f}")
print(f"t-test (for comparison): t={t_stat_raw:.3f}, p={p_t_raw:.4f}")
print()
print("For skewed protein data, Mann-Whitney U is more appropriate than the t-test.")

### Chi-Squared Test: Genotype Distributions

The chi-squared goodness-of-fit test checks whether observed categorical counts match expected proportions. In genetics, we test whether observed genotype frequencies match Hardy-Weinberg equilibrium expectations.

In [ ]:
# Chi-squared test: Hardy-Weinberg equilibrium
# SNP with allele frequency q(alt) = 0.3, p(ref) = 0.7
p_ref, q_alt = 0.7, 0.3
n_individuals = 500

# HWE expected frequencies: p², 2pq, q²
expected_freq = np.array([p_ref**2, 2*p_ref*q_alt, q_alt**2])
expected_counts = expected_freq * n_individuals

# Observed counts (slight deviation — possible selection pressure)
observed_counts = np.array([255, 200, 45])

chi2_stat, p_chi2 = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)

print("Hardy-Weinberg Equilibrium Test")
print(f"{'Genotype':<12} {'Expected':>10} {'Observed':>10}")
genotypes = ['AA (p²)', 'Aa (2pq)', 'aa (q²)']
for geno, exp, obs in zip(genotypes, expected_counts, observed_counts):
    print(f"{geno:<12} {exp:>10.1f} {obs:>10}")
print()
print(f"χ² = {chi2_stat:.4f}, df=2, p={p_chi2:.4f}")
if p_chi2 < 0.05:
    print("Significant departure from HWE — possible selection, population structure, or genotyping error.")
else:
    print("No significant departure from HWE.")

### Fisher's Exact Test: Association Testing

Fisher's exact test is used for 2×2 contingency tables, especially when sample sizes are small (expected cell counts < 5). In GWAS, it tests whether carrying an allele is associated with disease status.

In [ ]:
# Fisher's exact test: SNP association with disease
# 2x2 table: rows = disease/control, cols = alt allele present / absent
#                Alt allele   No alt allele
# Disease:          38            12
# Control:          22            28
contingency_table = np.array([[38, 12],
                               [22, 28]])

odds_ratio, p_fisher = stats.fisher_exact(contingency_table, alternative='two-sided')

print("Fisher's Exact Test: SNP association with disease")
print(f"\n  {'':15} {'Alt allele':>12} {'No alt allele':>14}")
print(f"  {'Disease cases':15} {contingency_table[0,0]:>12} {contingency_table[0,1]:>14}")
print(f"  {'Controls':15} {contingency_table[1,0]:>12} {contingency_table[1,1]:>14}")
print()
print(f"Odds ratio: {odds_ratio:.3f}")
print(f"p-value:    {p_fisher:.4f}")
print(f"The alt allele is {'associated' if p_fisher < 0.05 else 'not associated'} with disease (p={p_fisher:.3f}).")

### Checking Assumptions: Shapiro-Wilk and Levene's Test

Before applying parametric tests, verify:
1. **Normality**: Shapiro-Wilk test (H₀: data come from a normal distribution)
2. **Equal variances (homoscedasticity)**: Levene's test (H₀: group variances are equal)

A non-significant result (p > 0.05) means you *fail to reject* the assumption — it is consistent with the data.

In [ ]:
# Assumption checking before a two-sample t-test
group_A = np.random.normal(loc=7.5, scale=1.3, size=25)  # well-behaved
group_B = np.random.lognormal(mean=2.0, sigma=0.6, size=25)  # skewed

# Shapiro-Wilk normality test
sw_A, p_sw_A = stats.shapiro(group_A)
sw_B, p_sw_B = stats.shapiro(group_B)

# Levene's test for equal variances
lev_stat, p_lev = stats.levene(group_A, group_B)

print("--- Assumption Checking ---")
print(f"Shapiro-Wilk group A: W={sw_A:.4f}, p={p_sw_A:.4f} → {'Normal' if p_sw_A > 0.05 else 'Non-normal'}")
print(f"Shapiro-Wilk group B: W={sw_B:.4f}, p={p_sw_B:.4f} → {'Normal' if p_sw_B > 0.05 else 'Non-normal'}")
print(f"Levene's test:        F={lev_stat:.4f}, p={p_lev:.4f} → {'Equal variances' if p_lev > 0.05 else 'Unequal variances'}")
print()
if p_sw_A < 0.05 or p_sw_B < 0.05:
    print("Recommendation: use Mann-Whitney U (non-parametric) for group B.")
elif p_lev < 0.05:
    print("Recommendation: use Welch's t-test (unequal variances).")
else:
    print("Recommendation: Student's t-test is appropriate.")

---

## 4. Multiple Group Comparisons

When comparing more than two groups, running multiple pairwise t-tests inflates the Type I error rate. Instead, use:

- **One-way ANOVA**: parametric, tests whether any group mean differs
- **Kruskal-Wallis**: non-parametric alternative
- **Post-hoc tests**: determine *which* pairs differ (after a significant omnibus test)

### One-Way ANOVA: Expression Across Three Tissues

In [ ]:
# One-way ANOVA: comparing metabolite concentrations across three tissue types
# Using the expression data generated in Section 2
f_stat, p_anova = stats.f_oneway(gene_liver, gene_kidney, gene_brain)

print("--- One-Way ANOVA ---")
print(f"Liver mean:   {gene_liver.mean():.3f}  (n={len(gene_liver)})")
print(f"Kidney mean:  {gene_kidney.mean():.3f}  (n={len(gene_kidney)})")
print(f"Brain mean:   {gene_brain.mean():.3f}  (n={len(gene_brain)})")
print()
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value:     {p_anova:.4f}")
if p_anova < 0.05:
    print("At least one tissue group has a significantly different mean expression.")
    print("Proceed to post-hoc test to identify which pairs differ.")
else:
    print("No significant difference detected among tissue groups.")

### Kruskal-Wallis: Non-Parametric Alternative to ANOVA

When data are not normally distributed, Kruskal-Wallis tests whether the rank distributions differ across groups. It is the non-parametric equivalent of one-way ANOVA.

In [ ]:
# Kruskal-Wallis: non-parametric multi-group test
# Simulate skewed protein abundance data (log-normal)
prot_liver  = np.random.lognormal(mean=3.0, sigma=0.7, size=20)
prot_kidney = np.random.lognormal(mean=3.4, sigma=0.8, size=20)
prot_brain  = np.random.lognormal(mean=2.7, sigma=0.6, size=20)

h_stat, p_kw = stats.kruskal(prot_liver, prot_kidney, prot_brain)

print("--- Kruskal-Wallis Test ---")
print(f"Liver median:   {np.median(prot_liver):.2f}")
print(f"Kidney median:  {np.median(prot_kidney):.2f}")
print(f"Brain median:   {np.median(prot_brain):.2f}")
print()
print(f"H-statistic: {h_stat:.4f}")
print(f"p-value:     {p_kw:.4f}")
conclusion_kw = "Significant differences" if p_kw < 0.05 else "No significant differences"
print(f"{conclusion_kw} in protein abundance across tissues.")

### Post-Hoc Tests: Tukey HSD

When ANOVA is significant, Tukey's Honestly Significant Difference (HSD) test identifies which pairs of groups differ while controlling the family-wise error rate.

In [ ]:
# Tukey HSD post-hoc test with statsmodels
# Build a long-format DataFrame required by pairwise_tukeyhsd
all_expr = np.concatenate([gene_liver, gene_kidney, gene_brain])
all_labels = (['liver'] * len(gene_liver) +
               ['kidney'] * len(gene_kidney) +
               ['brain'] * len(gene_brain))

tukey_result = pairwise_tukeyhsd(endog=all_expr, groups=all_labels, alpha=0.05)
print(tukey_result.summary())

In [ ]:
# Visualise the Tukey confidence intervals
fig, ax = plt.subplots(figsize=(8, 4))
tukey_result.plot_simultaneous(ax=ax)
ax.set_title("Tukey HSD: 95% simultaneous confidence intervals")
ax.set_xlabel("Mean expression difference")
plt.tight_layout()
plt.show()
print("Intervals not crossing zero indicate significant pairwise differences.")

---

## 5. Correlation and Regression

### Pearson and Spearman Correlation

| Test | Assumes | Best for |
|------|---------|----------|
| Pearson | Linear relationship, bivariate normality | Continuous, approximately normal data |
| Spearman | Monotonic relationship (not necessarily linear) | Ordinal data, or continuous with outliers/skew |

We test whether mRNA expression predicts protein abundance — a common question in proteogenomics.

In [ ]:
# Pearson and Spearman correlation: mRNA vs protein levels
n_proteins = 80

# mRNA expression (predictor)
mrna_levels = np.random.normal(loc=8.0, scale=2.0, size=n_proteins)

# Protein levels: correlated with mRNA + independent noise
# True correlation coefficient ≈ 0.65
noise = np.random.normal(loc=0, scale=1.5, size=n_proteins)
protein_levels = 0.8 * mrna_levels + noise + 1.0

# Add a few outliers (common in proteomics data)
outlier_idx = [10, 25, 60]
protein_levels[outlier_idx] += np.random.normal(5, 1, size=3)

pearson_r, p_pearson  = stats.pearsonr(mrna_levels, protein_levels)
spearman_r, p_spearman = stats.spearmanr(mrna_levels, protein_levels)

print(f"Pearson r:  {pearson_r:.4f},  p={p_pearson:.4e}")
print(f"Spearman ρ: {spearman_r:.4f}, p={p_spearman:.4e}")
print()
print("Spearman is more robust to the outliers in protein data.")

In [ ]:
# Scatter plot with correlation coefficients annotated
fig, ax = plt.subplots(figsize=(8, 5))

# Regular points
regular = np.ones(n_proteins, dtype=bool)
regular[outlier_idx] = False
ax.scatter(mrna_levels[regular], protein_levels[regular],
           color='steelblue', alpha=0.7, s=35, label='Regular')
ax.scatter(mrna_levels[outlier_idx], protein_levels[outlier_idx],
           color='tomato', s=60, marker='D', label='Outliers', zorder=5)

# Regression line (on all data)
x_line = np.linspace(mrna_levels.min(), mrna_levels.max(), 100)
slope_ols = np.cov(mrna_levels, protein_levels)[0, 1] / np.var(mrna_levels)
intercept_ols = protein_levels.mean() - slope_ols * mrna_levels.mean()
ax.plot(x_line, slope_ols * x_line + intercept_ols, 'k--', lw=1.5, label='OLS line')

ax.set_xlabel('mRNA level (log2 TPM)')
ax.set_ylabel('Protein abundance (log2)')
ax.set_title(f'mRNA vs Protein: Pearson r={pearson_r:.3f}, Spearman ρ={spearman_r:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

### Simple Linear Regression with scipy and statsmodels

Linear regression models the relationship: **protein = β₀ + β₁ × mRNA + ε**

Use `scipy.stats.linregress` for a quick fit, or `statsmodels` for a full summary table with confidence intervals.

In [ ]:
# Simple linear regression with scipy (quick)
slope, intercept, r_value, p_slope, se_slope = stats.linregress(mrna_levels, protein_levels)

print("--- scipy.stats.linregress ---")
print(f"slope (β₁):   {slope:.4f}  (SE={se_slope:.4f}, p={p_slope:.4e})")
print(f"intercept:    {intercept:.4f}")
print(f"R²:           {r_value**2:.4f}")
print()

# Full regression with statsmodels (confidence intervals, residuals, diagnostics)
X_with_const = sm.add_constant(mrna_levels)  # adds the intercept column
ols_model = sm.OLS(protein_levels, X_with_const).fit()
print("--- statsmodels OLS summary (key rows) ---")
print(f"R-squared:    {ols_model.rsquared:.4f}")
print(f"Adj. R²:      {ols_model.rsquared_adj:.4f}")
print(f"F-statistic:  {ols_model.fvalue:.3f},  p={ols_model.f_pvalue:.4e}")
print()
print(ols_model.summary().tables[1])

### Regression Diagnostics: Residual Plots

After fitting a regression, always check the residuals:
- **Residuals vs fitted**: should show no pattern (homoscedasticity)
- **Q-Q of residuals**: should follow the diagonal (normality of errors)

Violations suggest the model is misspecified or a transformation is needed.

In [ ]:
# Regression diagnostics
fitted_values = ols_model.fittedvalues
residuals     = ols_model.resid

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 1. Residuals vs fitted values
ax1.scatter(fitted_values, residuals, color='steelblue', alpha=0.6, s=30)
ax1.axhline(0, color='black', lw=1, ls='--')
ax1.set_xlabel('Fitted values')
ax1.set_ylabel('Residuals')
ax1.set_title('Residuals vs Fitted')

# 2. Q-Q plot of residuals
(osm_r, osr_r), (slope_r, int_r, _) = stats.probplot(residuals, dist='norm')
ax2.scatter(osm_r, osr_r, color='steelblue', alpha=0.7, s=30)
ax2.plot(osm_r, slope_r * np.array(osm_r) + int_r, 'k--', lw=1.5)
ax2.set_xlabel('Theoretical quantiles')
ax2.set_ylabel('Residual quantiles')
ax2.set_title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.show()

# Shapiro-Wilk on residuals
sw_resid, p_sw_resid = stats.shapiro(residuals)
print(f"Shapiro-Wilk on residuals: W={sw_resid:.4f}, p={p_sw_resid:.4f}")
print(f"Residuals appear {'normal' if p_sw_resid > 0.05 else 'non-normal'} (α=0.05).")

### Multiple Regression: Gene Expression ~ Multiple Predictors

In biology, outcomes are rarely determined by a single variable. Multiple regression lets us model the effect of several predictors simultaneously while controlling for confounders.

In [ ]:
# Multiple regression: protein level ~ mRNA + GC_content + gene_length
n_genes_multi = 120

# Predictors
mrna_multi     = np.random.normal(8.0, 2.0, size=n_genes_multi)
gc_content     = np.random.uniform(0.35, 0.70, size=n_genes_multi)   # fraction
log_gene_len   = np.random.normal(7.5, 1.0, size=n_genes_multi)      # log(bp)

# True model: protein ~ 0.7*mRNA + 1.2*GC - 0.3*log_length + noise
noise_multi = np.random.normal(0, 1.0, size=n_genes_multi)
protein_multi = (0.7 * mrna_multi +
                 1.2 * gc_content * 10 +   # scale GC for interpretable coefficient
                 -0.3 * log_gene_len +
                 noise_multi + 5.0)

# Build DataFrame and fit with formula API
df_multi = pd.DataFrame({
    'protein': protein_multi,
    'mrna': mrna_multi,
    'gc_content': gc_content,
    'log_gene_len': log_gene_len
})

multi_model = smf.ols('protein ~ mrna + gc_content + log_gene_len', data=df_multi).fit()
print(multi_model.summary())

---

## 6. Bootstrap and Resampling

Resampling methods make no distributional assumptions. They estimate uncertainty directly from the data by repeatedly resampling.

### Bootstrap Confidence Intervals

The bootstrap estimates the sampling distribution of any statistic by:
1. Drawing B samples *with replacement* from the observed data (each sample same size as original)
2. Computing the statistic on each bootstrap sample
3. Using the resulting distribution to build confidence intervals

In [ ]:
# Bootstrap 95% confidence interval for median gene expression
# Small sample: only 15 tumor samples
tumor_small = np.random.normal(loc=9.2, scale=1.8, size=15)
n_bootstrap = 5000

bootstrap_medians = np.empty(n_bootstrap)
for i in range(n_bootstrap):
    resample = np.random.choice(tumor_small, size=len(tumor_small), replace=True)
    bootstrap_medians[i] = np.median(resample)

# Percentile method CI
ci_lower = np.percentile(bootstrap_medians, 2.5)
ci_upper = np.percentile(bootstrap_medians, 97.5)

print(f"Observed median:          {np.median(tumor_small):.4f}")
print(f"Bootstrap 95% CI:         [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Bootstrap SE of median:   {bootstrap_medians.std():.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(bootstrap_medians, bins=40, color='steelblue', alpha=0.7, density=True)
ax.axvline(np.median(tumor_small), color='black', lw=2, label='Observed median')
ax.axvline(ci_lower, color='tomato', lw=1.5, ls='--', label=f'95% CI [{ci_lower:.2f}, {ci_upper:.2f}]')
ax.axvline(ci_upper, color='tomato', lw=1.5, ls='--')
ax.set_xlabel('Bootstrap median')
ax.set_ylabel('Density')
ax.set_title('Bootstrap distribution of median expression')
ax.legend()
plt.tight_layout()
plt.show()

### Permutation Tests for Comparing Groups

A permutation test computes the exact null distribution of a test statistic by shuffling group labels many times. No normality assumption needed — only independence. This is equivalent to the exact randomisation test.

In [ ]:
# Permutation test: difference in means between two treatment groups
group_ctrl  = np.random.normal(loc=7.0, scale=1.2, size=20)
group_treat = np.random.normal(loc=8.1, scale=1.3, size=20)

observed_diff = group_treat.mean() - group_ctrl.mean()

# Pool all observations and permute group labels
combined = np.concatenate([group_ctrl, group_treat])
n_ctrl = len(group_ctrl)
n_perm = 10000

perm_diffs = np.empty(n_perm)
for i in range(n_perm):
    shuffled = np.random.permutation(combined)
    perm_diffs[i] = shuffled[n_ctrl:].mean() - shuffled[:n_ctrl].mean()

# Two-tailed p-value: fraction of permutations with |diff| >= |observed|
p_perm = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))

print(f"Observed difference: {observed_diff:.4f}")
print(f"Permutation p-value: {p_perm:.4f}")

# Compare with t-test
_, p_ttest = stats.ttest_ind(group_ctrl, group_treat)
print(f"t-test p-value:      {p_ttest:.4f}")
print("\nBoth approaches give similar p-values when data are approximately normal.")

### When to Use Resampling vs Parametric Tests

| Situation | Preferred approach |
|-----------|-------------------|
| Large sample, data approximately normal | Parametric (t-test, ANOVA) |
| Small sample, normality uncertain | Bootstrap or permutation test |
| Heavy tails or extreme outliers | Permutation test |
| Complex statistic (e.g., ratio, median) with no analytic SE | Bootstrap CI |
| Exact p-value needed for non-standard statistic | Permutation test |

**Key insight**: for large samples with approximately normal data, parametric and resampling methods agree closely. The resampling approach is more general but computationally heavier.

---

## 7. Power Analysis

**Statistical power** is the probability of detecting a true effect (1 - β). Before running an experiment, power analysis tells you how many samples you need. After the experiment, it tells you how reliably you could have detected the effect you observed.

The four quantities are linked — specify any three and solve for the fourth:
- **n**: sample size per group
- **effect_size**: Cohen's d = (μ₁ - μ₂) / σ_pooled
- **alpha**: significance level (Type I error rate)
- **power**: 1 - β (Type II error rate)

### Computing Required Sample Size

In [ ]:
# Sample size calculation for a two-sample t-test
# Biological question: how many tumor/normal samples to detect a 1.5 log2 fold change?
sigma_expr_assumed = 1.8   # typical SD of log2 expression
min_lfc = 1.5              # minimum log2 fold change to detect (biologically meaningful)
cohen_d = min_lfc / sigma_expr_assumed  # effect size

analysis = TTestIndPower()

# Required n for 80% power at alpha=0.05
n_required_80 = analysis.solve_power(effect_size=cohen_d, alpha=0.05, power=0.80)
# Required n for 90% power at alpha=0.05
n_required_90 = analysis.solve_power(effect_size=cohen_d, alpha=0.05, power=0.90)

print(f"Effect size (Cohen's d): {cohen_d:.3f}")
print(f"  (= {min_lfc} log2 FC / {sigma_expr_assumed} SD)")
print()
print(f"Required n per group for 80% power: {np.ceil(n_required_80):.0f}")
print(f"Required n per group for 90% power: {np.ceil(n_required_90):.0f}")
print()
print("Rule of thumb: you generally need at least 3 replicates, but 10+ for meaningful power.")

In [ ]:
# Power curve: achieved power as a function of sample size
sample_sizes = np.arange(3, 60)
power_values_80  = [analysis.solve_power(effect_size=cohen_d, alpha=0.05, nobs1=n) for n in sample_sizes]
power_values_small = [analysis.solve_power(effect_size=0.3, alpha=0.05, nobs1=n) for n in sample_sizes]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sample_sizes, power_values_80,   'steelblue', lw=2, label=f'Cohen\'s d={cohen_d:.2f} (LFC={min_lfc})')
ax.plot(sample_sizes, power_values_small, 'tomato',    lw=2, label='Cohen\'s d=0.3 (small effect)')
ax.axhline(0.80, ls='--', color='gray', lw=1, label='80% power threshold')
ax.axhline(0.90, ls=':',  color='gray', lw=1, label='90% power threshold')
ax.set_xlabel('Samples per group')
ax.set_ylabel('Statistical power')
ax.set_title('Power curves for two-sample t-test (α=0.05)')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---

## 8. Exercises

Work through these exercises to consolidate the module. Each exercise uses only the tools introduced above — no new packages needed.

### Exercise 1: Fit Distributions to Gene Expression Data

You are given `expr_obs`, a vector of log2 expression values from an RNA-seq experiment.

**Tasks:**
1. Use `stats.norm.fit(expr_obs)` to estimate μ and σ by MLE
2. Plot the empirical histogram overlaid with the fitted normal PDF
3. Run a Shapiro-Wilk test — is the normal model appropriate?
4. Compute P(expression > 11) under your fitted model

In [ ]:
# Exercise 1 — starter data
np.random.seed(7)
expr_obs = np.concatenate([
    np.random.normal(8.5, 1.1, size=80),
    np.random.normal(11.0, 0.6, size=20)   # small high-expression subgroup
])

# --- Your code here ---
# 1. Fit a normal distribution
# mu_fit, sigma_fit = stats.norm.fit(expr_obs)

# 2. Plot histogram + fitted PDF

# 3. Shapiro-Wilk test

# 4. P(expression > 11)

print("Uncomment the code above and implement each task.")

### Exercise 2: Compare Protein Levels Between Two Conditions

You have protein abundance measurements from untreated (`prot_ctrl`) and drug-treated (`prot_drug`) cells.

**Tasks:**
1. Visualise both distributions with overlapping histograms and box plots
2. Check normality (Shapiro-Wilk) and equal variances (Levene's)
3. Choose and apply the appropriate test (justify your choice)
4. Report the test statistic, p-value, and a one-sentence biological conclusion

In [ ]:
# Exercise 2 — starter data
np.random.seed(13)
prot_ctrl = np.random.lognormal(mean=4.5, sigma=0.9, size=22)
prot_drug = np.random.lognormal(mean=3.8, sigma=1.1, size=25)

# --- Your code here ---
# 1. Visualise distributions

# 2. Check assumptions

# 3. Apply appropriate test

# 4. Report results

print("Implement each task above.")

### Exercise 3: ANOVA on Metabolite Concentrations

Plasma concentrations of a metabolite were measured in four patient groups: healthy controls, pre-diabetic, type-2 diabetic, and type-1 diabetic.

**Tasks:**
1. Run a one-way ANOVA — is there a significant difference across groups?
2. If significant, run Tukey HSD to identify which pairs differ
3. Build a box plot annotated with the ANOVA p-value
4. What would you do differently if the data were heavily skewed?

In [ ]:
# Exercise 3 — starter data
np.random.seed(21)
healthy     = np.random.normal(5.2, 0.8, size=30)
prediabetic = np.random.normal(6.1, 1.0, size=28)
t2d         = np.random.normal(8.4, 1.5, size=35)
t1d         = np.random.normal(7.0, 1.3, size=25)

group_names = ['Healthy', 'Pre-diabetic', 'Type-2 DM', 'Type-1 DM']
groups      = [healthy, prediabetic, t2d, t1d]

# --- Your code here ---
# 1. One-way ANOVA

# 2. Tukey HSD if significant

# 3. Box plot with p-value annotation

# 4. (Written answer)

print("Implement each task above.")

### Exercise 4: Regression Analysis on Dose-Response Data

A drug was administered at six doses and the inhibitory effect on a kinase was measured in 5 replicates per dose.

**Tasks:**
1. Fit a simple linear regression: `response ~ log10(dose)`
2. Report R², the slope (with p-value), and the 95% confidence interval for the slope
3. Plot the data with the regression line and confidence band
4. Check residuals for normality and homoscedasticity
5. (Bonus) Compute a bootstrap 95% CI for the slope and compare to the OLS CI

In [ ]:
# Exercise 4 — starter data
np.random.seed(37)
doses_nM   = np.repeat([1, 10, 100, 1000, 10000, 100000], 5)
log_dose   = np.log10(doses_nM)

# True relationship: inhibition (%) = 15 * log10(dose) - 10 + noise
inhibition = 15 * log_dose - 10 + np.random.normal(0, 5, size=len(doses_nM))
inhibition = np.clip(inhibition, 0, 100)  # inhibition is bounded 0-100%

# --- Your code here ---
# 1. Fit linear regression

# 2. Report R², slope, CI

# 3. Plot with confidence band

# 4. Residual diagnostics

# 5. Bootstrap slope CI (bonus)

print("Implement each task above.")

---

## Summary

| Topic | Key function(s) | Biological use case |
|-------|----------------|--------------------|
| Normal distribution | `stats.norm(loc, scale)` | Log2 gene expression |
| Binomial distribution | `stats.binom(n, p)` | Variant allele read counts |
| Poisson distribution | `stats.poisson(mu)` | Somatic mutation counts |
| Negative binomial | `stats.nbinom(n, p)` | Overdispersed RNA-seq counts |
| Descriptive stats | `pd.DataFrame.describe()` | Summarising expression datasets |
| Normality check | `stats.shapiro()` | Assumption testing before t-test |
| Equal variance check | `stats.levene()` | Choose Student vs Welch t-test |
| One-sample t-test | `stats.ttest_1samp()` | Expression vs control reference |
| Two-sample t-test | `stats.ttest_ind()` | Tumor vs normal expression |
| Paired t-test | `stats.ttest_rel()` | Before/after treatment |
| Mann-Whitney U | `stats.mannwhitneyu()` | Non-parametric group comparison |
| Chi-squared | `stats.chisquare()` | Hardy-Weinberg equilibrium |
| Fisher's exact | `stats.fisher_exact()` | SNP-disease association |
| One-way ANOVA | `stats.f_oneway()` | Multi-tissue expression comparison |
| Kruskal-Wallis | `stats.kruskal()` | Non-parametric ANOVA |
| Tukey HSD | `pairwise_tukeyhsd()` | Post-hoc pairwise comparison |
| Pearson correlation | `stats.pearsonr()` | mRNA-protein correlation |
| Spearman correlation | `stats.spearmanr()` | Robust to outliers/skew |
| Simple regression | `sm.OLS()`, `stats.linregress()` | mRNA predicts protein level |
| Multiple regression | `smf.ols()` | Multi-predictor models |
| Bootstrap CI | manual `np.random.choice` loop | CI for any statistic |
| Permutation test | manual `np.random.permutation` loop | Exact null distribution |
| Power analysis | `TTestIndPower().solve_power()` | Sample size planning |

### Key Decision Rules

1. **Before any test**: check normality (Shapiro-Wilk) and equal variances (Levene's)
2. **Two groups, normal**: Welch's t-test (default; handles unequal variances)
3. **Two groups, non-normal**: Mann-Whitney U
4. **Three+ groups, normal**: one-way ANOVA → Tukey HSD if significant
5. **Three+ groups, non-normal**: Kruskal-Wallis → Dunn's test
6. **Association (2×2 table)**: Fisher's exact (small n) or chi-squared (large n)
7. **Small sample or complex statistic**: use bootstrap or permutation
8. **Always plan sample size in advance** — post-hoc power analysis is misleading

---

[← Previous: Biostatistics](../../Tier_0_Computational_Foundations/06_Biostatistics/01_biostatistics.ipynb) | [Next: Python Introduction →](../../Tier_1_Python_for_Bioinformatics/01_Python_Introduction/01_python_introduction.ipynb)

---